# Van der Pol Oscillator Tuning via Backpropagation through Harmonic Balance

This notebook shows how automatic differentiation through the Harmonic Balance solver
can solve oscillator design problems that are impractical with traditional tools.

## The problem

Given a Van der Pol oscillator (a nonlinear LC tank with negative resistance), tune its
parameters so the oscillation hits a precise target frequency and amplitude. This is a
joint nonlinear design problem with no closed-form solution.

**Traditional approach:** Sweep μ over a grid of values, run a transient simulation to
steady state for each, measure the amplitude, and read off the closest match. For a
two-parameter sweep (frequency + amplitude) the cost is quadratic in grid resolution.

**circulax approach:** `jax.grad` through the HB Newton solver via Optimistix's implicit
differentiation. The solver finds the periodic steady state in ~15 Newton steps; the
gradient is computed by differentiating through that fixed-point computation. A single
gradient call replaces the entire parameter sweep.

## Circuit: Van der Pol oscillator

The Van der Pol element has a cubic I–V characteristic:

$$I(V) = -\mu G_0 V + \frac{\mu G_0}{3} V^3$$

- For small $|V|$: slope $= -\mu G_0 < 0$ — negative resistance, oscillation grows
- Cubic term limits amplitude at $|V| \approx \sqrt{3/G_0}$ — stable limit cycle

The element is placed in parallel with an LC tank:

```
top_node ─── VDP   ─── GND    (Van der Pol: negative resistance + cubic saturation)
top_node ─── L1    ─── GND    (inductor)
top_node ─── C1    ─── GND    (capacitor)
top_node ─── Rdamp ─── GND    (small tank loss, models coil resistance)
```

The tank resonates at $f_0 = 1/(2\pi\sqrt{LC})$. The VDP element pumps energy in at
small amplitudes (negative conductance dominates) and absorbs energy at large amplitudes
(cubic term dominates), creating a stable limit cycle.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax

from circulax import compile_netlist, analyze_circuit, setup_harmonic_balance
from circulax.components.base_component import PhysicsReturn, Signals, States, component
from circulax.components.electronic import Capacitor, Inductor, Resistor
from circulax.utils import update_group_params, update_params_dict

# 64-bit precision is important: HB Newton requires accurate Jacobians, and
# gradient-based optimisation accumulates floating-point error across steps.
jax.config.update("jax_enable_x64", True)

plt.rcParams.update({
    "figure.figsize": (8, 3.5),
    "axes.grid": True,
    "lines.color": "grey",
    "patch.edgecolor": "grey",
    "text.color": "grey",
    "axes.facecolor": "white",
    "axes.edgecolor": "grey",
    "axes.labelcolor": "grey",
    "xtick.color": "grey",
    "ytick.color": "grey",
    "grid.color": "grey",
    "figure.facecolor": "white",
    "figure.edgecolor": "white",
    "savefig.facecolor": "white",
    "savefig.edgecolor": "white",
})

## Defining the Van der Pol component

Custom components are plain Python functions decorated with `@component`. The decorator
generates an Equinox module whose parameters (`mu`, `G0`) are JAX-traceable leaves,
making the component compatible with `jax.vmap`, `jax.jacfwd`, and `jax.grad`.

In [ ]:
@component(ports=("p1", "p2"))
def VanDerPolElement(signals: Signals, s: States, mu: float = 2.0, G0: float = 0.01) -> PhysicsReturn:
    """Nonlinear two-terminal element with cubic I-V characteristic.

    I(V) = -mu*G0*V + (mu*G0/3)*V^3

    The first term is a negative conductance (-mu*G0 < 0), which supplies
    energy to the tank when |V| is small and the circuit is below the limit
    cycle amplitude. The cubic term saturates the gain at large amplitudes,
    producing a stable oscillation.

    Limit-cycle amplitude estimate (fundamental, parallel tank):
        For mu >> G_tank/G0: A_fund ≈ 2*sqrt(mu) (normalised VdP result)
    With G0=0.01 and mu=2.0: A_fund ≈ 2*sqrt(2) ≈ 2.83 V
    """
    v = signals.p1 - signals.p2
    i = -mu * G0 * v + (mu * G0 / 3.0) * v**3
    return {"p1": i, "p2": -i}, {}


# Quick smoke-test: check that the I-V slope at V=0 is -mu*G0 (negative conductance)
vdp_test = VanDerPolElement(mu=2.0, G0=0.01)
f_test, _ = vdp_test(p1=0.1, p2=0.0)
print(f"VDP I at V=0.1 V : {f_test['p1']:.6f} A  (expected {-2.0*0.01*0.1 + (2.0*0.01/3)*0.1**3:.6f} A)")

# Verify negative conductance at small signal
small_v = 1e-3
f_small, _ = vdp_test(p1=small_v, p2=0.0)
g_small = f_small["p1"] / small_v
print(f"Small-signal conductance: {g_small:.4f} S  (expected {-2.0*0.01:.4f} S — negative")

## Building and compiling the circuit

All four elements connect between the same `top_node` and GND — they are in parallel.
The LC tank sets the oscillation frequency; `Rdamp` models the finite Q of a real inductor
(10 kΩ is intentionally large so tank losses are much smaller than the VDP negative
resistance, ensuring oscillation).

The tuple syntax for connections (`"GND,p1": ("VDP,p2", "C1,p2", ...)`) joins multiple
ports to the same node in a single line — equivalent to writing each pair separately.

In [ ]:
# ── Circuit parameters ────────────────────────────────────────────────────────
L_val  = 1e-6    # H  (1 µH)
C_val  = 1e-9    # F  (1 nF)
R_damp = 1e4     # Ω  (10 kΩ tank loss — small, Q ≈ R_damp * sqrt(C/L) ≈ 316)

f0 = 1.0 / (2.0 * np.pi * np.sqrt(L_val * C_val))
print(f"Tank resonant frequency : {f0 / 1e6:.4f} MHz")
print(f"Tank Q-factor (Rdamp)   : {R_damp * np.sqrt(C_val / L_val):.1f}")
print(f"VDP negative conductance: {-2.0 * 0.01:.4f} S  (at mu=2, G0=0.01)")
print(f"Tank loss conductance   : {1.0 / R_damp:.6f} S  (1/Rdamp)")
print(f"Net gain margin         : {abs(-2.0 * 0.01) / (1.0 / R_damp):.0f}×  >> 1, oscillation guaranteed")

# ── Netlist ──────────────────────────────────────────────────────────────────
# All four elements are in parallel between top_node and GND.
# GND is the special reserved name — any port touching "GND" is assigned node 0.
vdp_net = {
    "instances": {
        "VDP":   {"component": "vdp",       "settings": {"mu": 2.0,    "G0": 0.01}},
        "L1":    {"component": "inductor",   "settings": {"L": L_val}},
        "C1":    {"component": "capacitor",  "settings": {"C": C_val}},
        "Rdamp": {"component": "resistor",   "settings": {"R": R_damp}},
    },
    "connections": {
        # Connect all negative terminals to GND (node 0)
        "GND,p1":  ("VDP,p2", "L1,p2", "C1,p2", "Rdamp,p2"),
        # Connect all positive terminals to the same top_node
        "VDP,p1":  ("L1,p1", "C1,p1", "Rdamp,p1"),
    },
}

models = {
    "vdp":      VanDerPolElement,
    "inductor":  Inductor,
    "capacitor": Capacitor,
    "resistor":  Resistor,
}

# compile_netlist runs once: builds ComponentGroup objects with batched JAX arrays
# for parameters, and pre-computes index arrays for residual assembly.
groups, num_vars, net_map = compile_netlist(vdp_net, models)

print(f"\nSystem size : {num_vars} unknowns")
print(f"Node map    : {net_map}")
print(f"Groups      : {list(groups.keys())}")

# The oscillator node — all parallel elements share this port
osc_node = net_map["VDP,p1"]
print(f"Oscillator node index: {osc_node}")

# DC operating point: V=0 is the only fixed point (the VDP element has I(0)=0)
dc_solver = analyze_circuit(groups, num_vars, backend="dense")
y_dc = dc_solver.solve_dc(groups, jnp.zeros(num_vars))
print(f"DC operating point: max|y_dc| = {float(jnp.max(jnp.abs(y_dc))):.2e} V  (trivially zero)")

## Part 1 — Harmonic Balance finds the limit cycle

Transient simulation would need to integrate forward in time until the oscillation
envelope settles — often hundreds of RF cycles. Harmonic Balance finds the periodic
steady state **directly** by solving for the Fourier coefficients of the waveform.

`setup_harmonic_balance` builds a residual function over K = 2N+1 equally-spaced time
samples per period and solves it with Newton–Raphson (via Optimistix's
`FixedPointIteration` backed by `jax.lax.while_loop`). The result is JIT-compatible and
differentiable end-to-end.

In [ ]:
N_harm = 7   # 7 harmonics → K = 15 time points per period
K      = 2 * N_harm + 1

# Pass osc_node so setup_harmonic_balance automatically tries several sinusoidal
# initial amplitudes via jax.vmap and selects the limit-cycle solution.
# Users never need to choose a starting amplitude or think about the trivial y=0
# fixed point — the multi-start strategy handles it transparently.
run_hb = setup_harmonic_balance(groups, num_vars, freq=f0, num_harmonics=N_harm, osc_node=osc_node)
y_time, y_freq = jax.jit(run_hb)(y_dc)

print(f"y_time shape : {y_time.shape}  (K={K} time samples × {num_vars} nodes)")
print(f"y_freq shape : {y_freq.shape}  ({N_harm+1} harmonics × {num_vars} nodes)")

# y_freq[0] is the DC component, y_freq[1] is the fundamental, etc.
# Two-sided amplitude at harmonic k>=1 is 2 * |y_freq[k]|  (rfft folds negative freqs)
A_dc   = float(jnp.abs(y_freq[0, osc_node]))
A_fund = float(2.0 * jnp.abs(y_freq[1, osc_node]))
A_2nd  = float(2.0 * jnp.abs(y_freq[2, osc_node]))
A_3rd  = float(2.0 * jnp.abs(y_freq[3, osc_node]))

print(f"\nOscillator node (index {osc_node}) harmonic amplitudes:")
print(f"  DC (0f0)       : {A_dc:.4f} V")
print(f"  Fundamental f0 : {A_fund:.4f} V")
print(f"  2nd harmonic   : {A_2nd:.4f} V  ({A_2nd/A_fund*100:.1f}% of fundamental)")
print(f"  3rd harmonic   : {A_3rd:.4f} V  ({A_3rd/A_fund*100:.1f}% of fundamental)")
print(f"\nExpected amplitude (VdP limit cycle): ~2*sqrt(mu) = {2*np.sqrt(2.0):.3f} V")


In [ ]:
# ── Plot: time-domain waveform and harmonic spectrum ─────────────────────────
T     = 1.0 / f0
t_ns  = np.linspace(0.0, T * 1e9, K, endpoint=False)
v_osc = np.array(y_time[:, osc_node])

harmonics = np.arange(N_harm + 1)
scale     = np.where(harmonics == 0, 1.0, 2.0)  # fold negative-frequency energy
spectrum  = scale * np.abs(np.array(y_freq[:, osc_node]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left: time-domain waveform (one full period)
ax1.plot(t_ns, v_osc, "C0o-", ms=7, lw=1.5, label=f"$V_{{osc}}$ (HB, K={K})")
ax1.axhline(0, color="grey", lw=0.7, ls="--")
ax1.set_xlabel("Time (ns)")
ax1.set_ylabel("Voltage (V)")
ax1.set_title(f"Van der Pol limit cycle — $f_0$ = {f0/1e6:.3f} MHz")
ax1.legend(fontsize=9)

# Right: harmonic spectrum (bar chart)
ax2.bar(harmonics, spectrum, color="C0", alpha=0.85, edgecolor="white", linewidth=0.5)
ax2.set_xticks(harmonics)
ax2.set_xticklabels(["DC" if k == 0 else f"{k}$f_0$" for k in harmonics])
ax2.set_xlabel("Harmonic")
ax2.set_ylabel("Amplitude (V)")
ax2.set_title("Harmonic spectrum")

# Annotate the fundamental
ax2.annotate(
    f"{A_fund:.2f} V",
    xy=(1, A_fund), xytext=(1.6, A_fund * 0.85),
    fontsize=9, color="C0",
    arrowprops=dict(arrowstyle="->", color="C0", lw=0.8),
)

plt.tight_layout()
plt.show()

print("The odd-harmonic dominance (f0, 3f0, 5f0) is a hallmark of the symmetric cubic nonlinearity.")
print("Even harmonics (2f0, 4f0) are near-zero because I(-V) = -I(V) — the VDP element is odd.")

## Part 2 — Gradient-based amplitude tuning

Goal: find the value of μ such that the fundamental amplitude equals a target $A_{\text{target}}$.

The key insight: `setup_harmonic_balance` returns a function that is **end-to-end
differentiable** with respect to any JAX-traceable quantity captured in `groups`. The
parameter update is done with `update_group_params` (uses `eqx.tree_at` under the hood —
a pure functional update with no in-place mutation), so the entire loss computation is a
valid JAX program that `jax.grad` can differentiate through.

Optimistix's `FixedPointIteration` implements implicit differentiation: the gradient of
the fixed-point solution with respect to μ is computed via the implicit function theorem
rather than unrolling the Newton iterations.

In [ ]:
A_target = 1.5  # V — deliberately different from the default mu=2 result (~2.83 V)

# Use the Part 1 waveform as a warm start for the gradient-based optimisation.
# Inside jax.grad, the osc_node multi-start path goes through jnp.argmax, which
# has zero gradient and corrupts dA/dmu.  A fixed y_flat_init avoids argmax
# entirely: Optimistix's implicit diff then computes the exact gradient through
# the fixed-point equation alone.
y_flat_warmstart = y_time.flatten()  # shape (K * num_vars,) — good starting waveform for nearby mu


def loss_fn_mu(mu: jax.Array) -> jax.Array:
    """Squared error between fundamental amplitude and target, as a function of mu."""
    groups_new = update_group_params(groups, "vdp", "mu", mu)
    run_hb_new = setup_harmonic_balance(groups_new, num_vars, freq=f0, num_harmonics=N_harm)
    _, y_freq_new = run_hb_new(y_dc, y_flat_init=y_flat_warmstart)
    A_fund = 2.0 * jnp.abs(y_freq_new[1, osc_node])
    return (A_fund - A_target) ** 2


# Test: evaluate loss and gradient at the default mu=2.0
mu_test = jnp.array(2.0)
loss_val, grad_val = jax.value_and_grad(loss_fn_mu)(mu_test)
print(f"At mu=2.0:  loss = {float(loss_val):.4f} V²,  grad = {float(grad_val):.4f} V²/[mu]")
print(f"  Current amplitude ≈ {float(jnp.sqrt(loss_val)) + A_target:.3f} V,  target = {A_target} V")
print(f"  Negative gradient → decreasing mu reduces amplitude (as expected)")

# ── Gradient descent — scalar parameter, no need for Adam ───────────────────
mu = jnp.array(3.0)  # start above the solution to test convergence from above
lr = 0.05

mu_history    = [float(mu)]
loss_history  = []

val_and_grad_jit = jax.jit(jax.value_and_grad(loss_fn_mu))

print(f"\nGradient descent (lr={lr}, starting from mu={float(mu):.1f}):")
for step in range(80):
    loss, g = val_and_grad_jit(mu)
    loss_history.append(float(loss))
    mu = mu - lr * g
    mu_history.append(float(mu))
    if step % 20 == 0 or step == 79:
        # Standalone evaluation uses osc_node (multi-start) so it always finds the limit cycle
        groups_cur = update_group_params(groups, "vdp", "mu", mu)
        _, yf_cur  = jax.jit(setup_harmonic_balance(groups_cur, num_vars, freq=f0, num_harmonics=N_harm, osc_node=osc_node))(y_dc)
        A_cur = float(2.0 * jnp.abs(yf_cur[1, osc_node]))
        print(f"  Step {step:3d}: mu = {float(mu):.4f},  loss = {float(loss):.6f},  A_fund = {A_cur:.4f} V")

mu_opt = float(mu)
print(f"\nOptimised mu = {mu_opt:.4f}")


In [ ]:
# ── Compute waveforms before and after optimisation ───────────────────────────
groups_before = update_group_params(groups, "vdp", "mu", jnp.array(2.0))
groups_after  = update_group_params(groups, "vdp", "mu", jnp.array(mu_opt))

_, yf_before = jax.jit(setup_harmonic_balance(groups_before, num_vars, freq=f0, num_harmonics=N_harm, osc_node=osc_node))(y_dc)
yt_after, yf_after = jax.jit(setup_harmonic_balance(groups_after, num_vars, freq=f0, num_harmonics=N_harm, osc_node=osc_node))(y_dc)

A_before = float(2.0 * jnp.abs(yf_before[1, osc_node]))
A_after  = float(2.0 * jnp.abs(yf_after[1, osc_node]))

v_before = np.array(y_time[:, osc_node])   # mu=2 waveform from Part 1
v_after  = np.array(yt_after[:, osc_node])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left: time-domain comparison
ax1.plot(t_ns, v_before, "C0-",  lw=2,   label=f"μ = 2.00  →  A = {A_before:.2f} V (initial)")
ax1.plot(t_ns, v_after,  "C1--", lw=2.5, label=f"μ = {mu_opt:.2f}  →  A = {A_after:.2f} V (optimised)")
ax1.axhline( A_target, color="grey", lw=0.8, ls=":", label=f"Target = {A_target} V")
ax1.axhline(-A_target, color="grey", lw=0.8, ls=":")
ax1.set_xlabel("Time (ns)")
ax1.set_ylabel("Voltage (V)")
ax1.set_title("Before and after μ tuning")
ax1.legend(fontsize=9)

# Right: loss convergence
ax2.semilogy(loss_history, "C0", lw=2)
ax2.set_xlabel("Gradient descent step")
ax2.set_ylabel("Loss  $(A_{\\mathrm{fund}} - A_{\\mathrm{target}})^2$  (V²)")
ax2.set_title(f"Amplitude tuning convergence (target {A_target} V)")

plt.tight_layout()
plt.show()

print(f"Amplitude error after optimisation: {abs(A_after - A_target)*1000:.2f} mV")


## Part 3 — Joint tuning: frequency and amplitude

Now we solve the harder problem: simultaneously tune L, C, and μ so the oscillator
hits a **new target frequency** (8 MHz, shifted from 5 MHz) **and** a target amplitude
(1.0 V).

This is a joint nonlinear design problem. The traditional approach would be a 3D sweep
over (L, C, μ) — thousands of HB solves just to map the landscape. With `jax.grad` we
get the exact gradient with one HB solve, and Adam navigates directly to the solution.

**Log-space parameterisation** keeps all three parameters positive and balances their
gradients: a 10% change in L feels the same as a 10% change in μ, regardless of the
absolute scale. We also pass the resonant frequency (derived from L and C) directly to
`setup_harmonic_balance` — this ensures the HB discretisation always matches the actual
oscillation frequency, which is essential for convergence when L and C are changing.

In [ ]:
f_target = 8e6    # Hz — 8 MHz target (up from ~5 MHz)
A_target_j = 1.0  # V  — 1 V fundamental amplitude

LC_target = 1.0 / (2.0 * np.pi * f_target) ** 2
print(f"Target frequency    : {f_target/1e6:.1f} MHz")
print(f"Required L*C product: {LC_target:.3e} H·F")
print(f"Starting L={L_val*1e6:.2f} µH, C={C_val*1e9:.2f} nF  → f0={f0/1e6:.3f} MHz")

# Fixed sinusoidal warm start for the joint optimisation loss — amplitude 2 V at osc_node,
# all other nodes zero.  This is above the basin-of-attraction saddle (~1.15 V) for all
# (L, C, mu) values explored during optimisation.  Using a constant here keeps the
# gradient path free of argmax, so Optimistix's implicit diff works cleanly.
_phase_k = 2.0 * jnp.pi * jnp.arange(K, dtype=jnp.float64) / K
y_flat_hb_init = (
    jnp.zeros(K * num_vars, dtype=jnp.float64)
    .at[jnp.arange(K) * num_vars + osc_node].set(2.0 * jnp.sin(_phase_k))
)


def loss_joint(log_params: jax.Array) -> jax.Array:
    """Joint loss over (log_L, log_C, log_mu) for frequency + amplitude targets."""
    log_L, log_C, log_mu = log_params
    L   = jnp.exp(log_L)
    C   = jnp.exp(log_C)
    mu  = jnp.exp(log_mu)

    f_resonant = 1.0 / (2.0 * jnp.pi * jnp.sqrt(L * C))

    grps = update_params_dict(groups, "inductor",  "L1", "L", L)
    grps = update_params_dict(grps,   "capacitor", "C1", "C", C)
    grps = update_group_params(grps,  "vdp", "mu", mu)

    run_hb_j = setup_harmonic_balance(grps, num_vars, freq=f_resonant, num_harmonics=N_harm)
    _, y_freq_j = run_hb_j(jnp.zeros(num_vars), y_flat_init=y_flat_hb_init)

    A_fund = 2.0 * jnp.abs(y_freq_j[1, osc_node])

    loss_freq = ((f_resonant - f_target) / f_target) ** 2
    loss_amp  = ((A_fund - A_target_j) / A_target_j) ** 2

    return loss_freq + loss_amp


# Sanity check
log_params_0 = jnp.log(jnp.array([L_val, C_val, 2.0]))
loss_0 = loss_joint(log_params_0)
print(f"\nInitial joint loss: {float(loss_0):.4f}  (freq error + amplitude error)")

# ── Adam optimisation ─────────────────────────────────────────────────────────
optimizer   = optax.adam(0.05)
log_params  = log_params_0
opt_state   = optimizer.init(log_params)
losses_joint = []
param_log_hist = [np.array(log_params)]

val_grad_joint = jax.jit(jax.value_and_grad(loss_joint))

print(f"\nAdam optimisation (200 steps, lr=0.05):")
for i in range(200):
    loss, grads = val_grad_joint(log_params)
    losses_joint.append(float(loss))
    updates, opt_state = optimizer.update(grads, opt_state)
    log_params = optax.apply_updates(log_params, updates)
    param_log_hist.append(np.array(log_params))
    if i % 50 == 0 or i == 199:
        L_c, C_c, mu_c = np.exp(np.array(log_params))
        f_c_cur = 1.0 / (2.0 * np.pi * np.sqrt(L_c * C_c))
        print(f"  Step {i:3d}: loss={float(loss):.5f},  f={f_c_cur/1e6:.3f} MHz,  L={L_c*1e6:.3f} µH,  C={C_c*1e9:.3f} nF,  mu={mu_c:.3f}")

L_opt, C_opt, mu_opt_j = np.exp(np.array(log_params))
f_opt = 1.0 / (2.0 * np.pi * np.sqrt(L_opt * C_opt))
print(f"\nFinal: f={f_opt/1e6:.4f} MHz  (target {f_target/1e6:.1f} MHz),  L={L_opt*1e6:.4f} µH,  C={C_opt*1e9:.4f} nF,  mu={mu_opt_j:.4f}")


In [ ]:
# ── Evaluate the optimised waveform ──────────────────────────────────────────
grps_opt = update_params_dict(groups, "inductor",  "L1", "L", L_opt)
grps_opt = update_params_dict(grps_opt, "capacitor", "C1", "C", C_opt)
grps_opt = update_group_params(grps_opt, "vdp", "mu", jnp.array(mu_opt_j))

run_hb_opt = setup_harmonic_balance(grps_opt, num_vars, freq=f_opt, num_harmonics=N_harm, osc_node=osc_node)
yt_opt, yf_opt = jax.jit(run_hb_opt)(jnp.zeros(num_vars))

A_fund_opt = float(2.0 * jnp.abs(yf_opt[1, osc_node]))
T_opt      = 1.0 / f_opt
t_opt_ns   = np.linspace(0.0, T_opt * 1e9, K, endpoint=False)
v_opt      = np.array(yt_opt[:, osc_node])

# Also build a reference waveform at the initial parameters for comparison
grps_init  = update_params_dict(groups, "inductor",  "L1", "L", L_val)
grps_init  = update_params_dict(grps_init, "capacitor", "C1", "C", C_val)
grps_init  = update_group_params(grps_init, "vdp", "mu", jnp.array(2.0))
run_hb_init = setup_harmonic_balance(grps_init, num_vars, freq=f0, num_harmonics=N_harm, osc_node=osc_node)
yt_init, _ = jax.jit(run_hb_init)(jnp.zeros(num_vars))
v_init = np.array(yt_init[:, osc_node])

param_hist = np.exp(np.array(param_log_hist))  # shape (201, 3): [L, C, mu]
f_hist = 1.0 / (2.0 * np.pi * np.sqrt(param_hist[:, 0] * param_hist[:, 1]))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ── Left: loss convergence ────────────────────────────────────────────────────
ax = axes[0]
ax.semilogy(losses_joint, "C0", lw=2)
ax.set_xlabel("Adam step")
ax.set_ylabel("Loss (freq² + amp²)")
ax.set_title("Joint optimisation convergence")

# ── Middle: frequency and amplitude trajectories ──────────────────────────────
ax = axes[1]
steps = np.arange(len(f_hist))
ax2r = ax.twinx()
l1, = ax.plot(steps, f_hist / 1e6, "C0", lw=2, label="frequency (MHz)")
ax.axhline(f_target / 1e6, color="C0", ls="--", lw=1, alpha=0.6, label="f target")
l3, = ax2r.plot(steps, param_hist[:, 2], "C1", lw=2, label="μ")
ax2r.axhline(mu_opt_j, color="C1", ls="--", lw=1, alpha=0.6)
ax2r.set_ylabel("μ", color="C1")
ax2r.tick_params(axis="y", labelcolor="C1")
ax.set_xlabel("Adam step")
ax.set_ylabel("Frequency (MHz)", color="C0")
ax.tick_params(axis="y", labelcolor="C0")
ax.set_title("Parameter trajectories")
lines = [l1, l3]
ax.legend(lines, [l.get_label() for l in lines], fontsize=8, loc="upper right")

# ── Right: initial vs optimised waveforms ─────────────────────────────────────
ax = axes[2]
# Normalise time axis to show one full period for each waveform
t_init_norm = np.linspace(0, 1, K, endpoint=False)
t_opt_norm  = np.linspace(0, 1, K, endpoint=False)
ax.plot(t_init_norm, v_init, "C2-",  lw=2,   label=f"Initial: f={f0/1e6:.2f} MHz, A={float(2*jnp.abs(yf_opt[1,osc_node])):.2f} V")
ax.plot(t_opt_norm,  v_opt,  "C0--", lw=2.5, label=f"Optimised: f={f_opt/1e6:.2f} MHz, A={A_fund_opt:.2f} V")
ax.axhline( A_target_j, color="grey", lw=0.8, ls=":", label=f"Target amplitude ±{A_target_j} V")
ax.axhline(-A_target_j, color="grey", lw=0.8, ls=":")
ax.set_xlabel("Normalised time (t / T)")
ax.set_ylabel("Voltage (V)")
ax.set_title("Waveform: initial vs optimised")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f"\nFinal results:")
print(f"  Frequency : {f_opt/1e6:.4f} MHz  (target {f_target/1e6:.1f} MHz,  error {abs(f_opt - f_target)/f_target*100:.2f}%)")
print(f"  Amplitude : {A_fund_opt:.4f} V    (target {A_target_j:.1f} V,        error {abs(A_fund_opt - A_target_j)/A_target_j*100:.2f}%)")
print(f"  L = {L_opt*1e6:.4f} µH,  C = {C_opt*1e9:.4f} nF,  mu = {mu_opt_j:.4f}")


## Summary

Starting from a Van der Pol oscillator running at ~5 MHz with ~2.8 V amplitude, gradient
descent simultaneously tuned L, C, and μ to hit 8 MHz and 1.0 V in 200 Adam steps —
with no grid sweeps and no manual iteration.

### What made this possible?

| Step | Tool | Role |
|------|------|------|
| Component definition | `@component` decorator | Generates JAX-traceable Equinox module |
| Netlist compilation | `compile_netlist` | Runs once; produces vmappable `ComponentGroup` objects |
| Differentiable parameter update | `update_params_dict` / `update_group_params` | `eqx.tree_at` functional update; no recompilation |
| Periodic steady state | `setup_harmonic_balance` | Finds limit cycle in ~15 Newton steps; JIT-compatible |
| Exact gradients | `jax.grad` | Differentiates through the HB Newton solver via implicit differentiation |
| Optimisation | `optax.adam` | First-order optimiser in log-space; 200 steps to convergence |

### Going further

The same pattern extends directly to:

- **Phase noise minimisation** — add a noise source and minimise the HB-estimated phase
  noise spectral density as a function of tank Q and bias point.
- **Injection locking** — add a small-signal source and tune the free-running frequency
  to lock at the injection frequency with maximum locking range.
- **Coupled oscillator arrays** — vmap the HB solve over an array of weakly coupled
  oscillators and optimise the coupling network for synchronisation.
- **CMOS VCO design** — replace the VDP element with a transistor-level cross-coupled
  pair (using NMOS / PMOS components from `circulax.components.electronic`) and
  sweep the varactor capacitance as a differentiable parameter.